# Introduction

This notebook prepares the dataset and feature inputs required for the sentiment analysis models used in the project. The goal is to transform the cleaned Amazon review dataset into a structured format that can be used by multiple machine learning models.

The notebook begins by loading the processed dataset containing the review text and associated metadata such as product ratings, helpful votes, and verification status. Since the dataset originally contains star ratings, these ratings are converted into sentiment labels. Reviews with ratings of 4 or 5 stars are classified as positive, while ratings below 4 are classified as negative.

Additional metadata features are also created to enrich the dataset. For example, the review length feature is generated to capture the number of words in each review, and the verified purchase status is converted into a numeric format so that it can be used by machine learning algorithms.

Next, the dataset is separated into features (X) and target labels (y), after which it is split into training and testing sets using an 80/20 split. This ensures that the models can be trained on one portion of the data and evaluated on unseen data.

To convert the textual reviews into numerical representations, the notebook applies TF-IDF vectorization, which transforms the review text into a matrix of weighted word features.

Finally, the generated TF-IDF vectorizer and transformed datasets are saved as shared files. These saved files allow multiple model notebooks in the project to use the same preprocessed features, ensuring consistency across experiments.

In [ ]:
# Import Neccesary Libraries

import pandas as pd
import numpy as np
import joblib
import os

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

### Load Dataset
Since preprocessing was already done during the data cleaning phase, load the dataset containing
reviewText_processed
overall
vote
verified

In [22]:
#load and preview the data
df = pd.read_csv("amazon_reviews_processed.csv")

df.head()

/var/folders/0m/7cd4sg3s2h9_28bz90nk68wr0000gn/T/ipykernel_51678/3455308728.py:2: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("amazon_reviews_processed.csv")


,asin,category,reviewerID,reviewerName,overall,reviewText,unixReviewTime,reviewTime,verified,vote,has_vote,review_length,word_count,short_review_flag,reviewText_cleaned,reviewText_processed
0,B00CYQP3AK,Amazon Devices & Accessories,A3UXVNOFC8TJPT,Bkonasek,5.0,I received this as a birthday present. I am do...,1382140800,"10 19, 2013",False,2,1,130,22,0,I received this as a birthday present I am do ...,receive birthday present pleased product pictu...
1,B00CYQP3AK,Amazon Devices & Accessories,AC2VR9U3NN2UD,tikitoo,2.0,Battery will not charge to 100%. When I receiv...,1382140800,"10 19, 2013",True,22,1,1693,325,0,Battery will not charge to 100 When I received...,battery not charge receive device hour charger...
2,B00CYQP3AK,Amazon Devices & Accessories,A14D0LJ1YRLLCI,gabbie03,1.0,Buyer Beware. these units are being shipped ri...,1382140800,"10 19, 2013",True,54,1,1051,198,0,Buyer Beware these units are being shipped rig...,buyer beware unit ship right app netflix unit ...
3,B00CYQP3AK,Amazon Devices & Accessories,A3FVF1JIOA5GEK,MN_Ranger,5.0,"Netflix Issue:\nWhen I first got the KFHDX, I ...",1382054400,"10 18, 2013",True,577,1,31090,5690,0,Netflix Issue When I first got the KFHDX I had...,netflix issue get kfhdx no problem netflix day...
4,B00CYQP3AK,Amazon Devices & Accessories,A3KP101J8H7EWY,L.D.K,5.0,"I have owned many versions of the Kindle, and ...",1382054400,"10 18, 2013",True,8,1,311,58,0,I have owned many versions of the Kindle and t...,own version kindle far good interface friendly...


### Generate Sentiment Labels from Ratings
Convert the star rating into sentiment classes.

In [23]:
def rating_sentiment(rating):

    if rating >= 4:
        return "positive"

    else:
        return "negative"

df["sentiment_label"] = df["overall"].apply(rating_sentiment)

In [24]:
#Check distribution
df["sentiment_label"].value_counts()

sentiment_label
positive    710417
negative    116981
Name: count, dtype: int64

### Create Metadata Features

In [25]:
#Review Length
df["review_length"] = df["reviewText_processed"].str.split().str.len()

In [26]:
df.head()

,asin,category,reviewerID,reviewerName,overall,reviewText,unixReviewTime,reviewTime,verified,vote,has_vote,review_length,word_count,short_review_flag,reviewText_cleaned,reviewText_processed,sentiment_label
0,B00CYQP3AK,Amazon Devices & Accessories,A3UXVNOFC8TJPT,Bkonasek,5.0,I received this as a birthday present. I am do...,1382140800,"10 19, 2013",False,2,1,11,22,0,I received this as a birthday present I am do ...,receive birthday present pleased product pictu...,positive
1,B00CYQP3AK,Amazon Devices & Accessories,AC2VR9U3NN2UD,tikitoo,2.0,Battery will not charge to 100%. When I receiv...,1382140800,"10 19, 2013",True,22,1,133,325,0,Battery will not charge to 100 When I received...,battery not charge receive device hour charger...,negative
2,B00CYQP3AK,Amazon Devices & Accessories,A14D0LJ1YRLLCI,gabbie03,1.0,Buyer Beware. these units are being shipped ri...,1382140800,"10 19, 2013",True,54,1,86,198,0,Buyer Beware these units are being shipped rig...,buyer beware unit ship right app netflix unit ...,negative
3,B00CYQP3AK,Amazon Devices & Accessories,A3FVF1JIOA5GEK,MN_Ranger,5.0,"Netflix Issue:\nWhen I first got the KFHDX, I ...",1382054400,"10 18, 2013",True,577,1,2661,5690,0,Netflix Issue When I first got the KFHDX I had...,netflix issue get kfhdx no problem netflix day...,positive
4,B00CYQP3AK,Amazon Devices & Accessories,A3KP101J8H7EWY,L.D.K,5.0,"I have owned many versions of the Kindle, and ...",1382054400,"10 18, 2013",True,8,1,23,58,0,I have owned many versions of the Kindle and t...,own version kindle far good interface friendly...,positive


### Convert Verified Purchase

Why Convert verified to Numbers?

Most machine learning models (like Logistic Regression or Support Vector Machine) cannot use text or boolean values directly. They require numeric inputs.

In [27]:

df["verified"] = df["verified"].astype(int)

In [28]:
df["verified"].unique()

array([0, 1])

1 - Not verified

0 - verified

In [29]:
df.head()

,asin,category,reviewerID,reviewerName,overall,reviewText,unixReviewTime,reviewTime,verified,vote,has_vote,review_length,word_count,short_review_flag,reviewText_cleaned,reviewText_processed,sentiment_label
0,B00CYQP3AK,Amazon Devices & Accessories,A3UXVNOFC8TJPT,Bkonasek,5.0,I received this as a birthday present. I am do...,1382140800,"10 19, 2013",0,2,1,11,22,0,I received this as a birthday present I am do ...,receive birthday present pleased product pictu...,positive
1,B00CYQP3AK,Amazon Devices & Accessories,AC2VR9U3NN2UD,tikitoo,2.0,Battery will not charge to 100%. When I receiv...,1382140800,"10 19, 2013",1,22,1,133,325,0,Battery will not charge to 100 When I received...,battery not charge receive device hour charger...,negative
2,B00CYQP3AK,Amazon Devices & Accessories,A14D0LJ1YRLLCI,gabbie03,1.0,Buyer Beware. these units are being shipped ri...,1382140800,"10 19, 2013",1,54,1,86,198,0,Buyer Beware these units are being shipped rig...,buyer beware unit ship right app netflix unit ...,negative
3,B00CYQP3AK,Amazon Devices & Accessories,A3FVF1JIOA5GEK,MN_Ranger,5.0,"Netflix Issue:\nWhen I first got the KFHDX, I ...",1382054400,"10 18, 2013",1,577,1,2661,5690,0,Netflix Issue When I first got the KFHDX I had...,netflix issue get kfhdx no problem netflix day...,positive
4,B00CYQP3AK,Amazon Devices & Accessories,A3KP101J8H7EWY,L.D.K,5.0,"I have owned many versions of the Kindle, and ...",1382054400,"10 18, 2013",1,8,1,23,58,0,I have owned many versions of the Kindle and t...,own version kindle far good interface friendly...,positive


### Define Feature Set

Features will include:
- review text
- review_length
- vote
- verified

In [30]:
X = df[["reviewText_processed", "review_length", "vote", "verified"]]

y = df["sentiment_label"]

## Train/Test Split

In [31]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

The dataset was split into training and testing sets using train_test_split, where 80% of the data was used for training the model and 20% was reserved for testing. The training set allows the model to learn patterns from the data, while the test set is used to evaluate the model’s performance on unseen data. A random_state of 42 was set to ensure reproducibility of the data spli

### Generate TF-IDF Features

In [32]:
tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1,2)
)

X_train_text = tfidf.fit_transform(X_train["reviewText_processed"])

X_test_text = tfidf.transform(X_test["reviewText_processed"])

### save shared Data

These files will be used by all model notebooks

In [50]:
# create folder if it doesn't exist
os.makedirs("shared", exist_ok=True)

joblib.dump(tfidf, "shared/tfidf_vectorizer.pkl")

joblib.dump(X_train_text, "shared/X_train_text.pkl")
joblib.dump(X_test_text, "shared/X_test_text.pkl")

joblib.dump(X_train[["review_length","vote","verified"]], "shared/X_train_meta.pkl")
joblib.dump(X_test[["review_length","vote","verified"]], "shared/X_test_meta.pkl")

joblib.dump(y_train, "shared/y_train.pkl")
joblib.dump(y_test, "shared/y_test.pkl")


print("Done.")

Done.
